<a href="https://colab.research.google.com/github/Storm00212/Data-science-and-ml-resource/blob/main/Simple_snn_framework.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
!pip uninstall -y snntorch-ipu
!pip install snntorch

Found existing installation: snntorch-ipu 0.5.18
Uninstalling snntorch-ipu-0.5.18:
  Successfully uninstalled snntorch-ipu-0.5.18
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.6/125.6 kB 8.0 MB/s eta 0:00:00


In [5]:
import torch, torch.nn as nn
import snntorch as snn
import snntorch.functional as SF

In [7]:
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

batch_size = 128
data_path='/tmp/data/mnist'

# Define a transform
transform = transforms.Compose([
            transforms.Resize((28, 28)),
            transforms.Grayscale(),
            transforms.ToTensor(),
            transforms.Normalize((0,), (1,))])

mnist_train = datasets.MNIST(data_path, train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(data_path, train=False, download=True, transform=transform)

# Create DataLoaders
train_loader = DataLoader(dataset=mnist_train, batch_size=batch_size, shuffle=True, num_workers=2)
test_loader = DataLoader(dataset=mnist_test, batch_size=batch_size, shuffle=True, num_workers=2)

In [9]:
class Net(nn.Module):
    def __init__(self):
        super().__init__()

        num_inputs = 28*28
        num_hidden = 1000
        num_outputs = 10

        # Initialize layers
        self.fc1 = nn.Linear(num_inputs, num_hidden)
        self.lif1 = snn.LIF(beta=0.9)
        self.fc2 = nn.Linear(num_hidden, num_outputs)

    def forward(self, x):
        # Flatten input
        x = x.view(x.size(0), -1)

        # Initialize hidden states
        mem1 = self.lif1.init_hidden()

        # Pass data through layers
        cur1 = self.fc1(x)
        spk1, mem1 = self.lif1(cur1, mem1)
        cur2 = self.fc2(spk1)

        return cur2

# Initialize the network
net = Net()
print(net)

Net(
  (fc1): Linear(in_features=784, out_features=1000, bias=True)
  (lif1): LIF()
  (fc2): Linear(in_features=1000, out_features=10, bias=True)
)
